# Ceiling Diagnostics + Diffusion Go/No-Go
**সম্পূর্ণ self-contained — কোনো আগের cell লাগবে না।**

### Kaggle-এ চালানোর আগে:
1. **Add Data** → pretrained weight (`inf_model_007_256_resnet.h5`) attach করো
2. **Run All**

### কী কী পরীক্ষা হবে:
1. GT-blob ceiling vs **σ** (0.10 → 0.02) — বিনামূল্যে, কোনো training লাগে না
2. GT-blob ceiling vs **M** (256 vs 512)
3. Missed source-দের **sin(ψ)** — Jacobian hypothesis check
4. **Diffusion go/no-go**: blob vs sharpened vs argmax (আসল ResNet output দিয়ে)

In [ ]:
# Cell 1 — Setup
import importlib, subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)
    import cv2

import os, math, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import maximum_filter

np.random.seed(42)
print('✅ Ready')

In [ ]:
# Cell 2 — Physics + Data Generator (paper-exact, self-contained)
class _H(np.ndarray):
    @property
    def H(self): return self.conj().transpose()

def ev(n, angle):
    return ((1/np.sqrt(n)) * np.exp(-1j*np.pi*np.cos(angle)*np.arange(n))).reshape(-1,1)

def make_F(P, nt):
    phi = np.arccos((1/np.pi)*np.angle(np.exp( 1j*(2*np.pi/P)*np.arange(P))))
    F = np.zeros((nt, P), dtype=complex)
    for i, ph in enumerate(phi): F[:, i] = ev(nt, ph).ravel()
    return F

def make_W(Q, nr):
    phi = np.arccos((1/np.pi)*np.angle(np.exp(-1j*(2*np.pi/Q)*np.arange(Q))))
    W = np.zeros((nr, Q), dtype=complex)
    for i, ph in enumerate(phi): W[:, i] = ev(nr, ph).ravel()
    return W

def gen_channel(nr, nt, phi_l, psi_l, alpha_l):
    H = np.zeros((nr, nt), dtype=complex)
    for a, phi, psi in zip(alpha_l, phi_l, psi_l):
        H += a * (ev(nr, psi) * ev(nt, phi).view(_H).H)
    return np.sqrt(nt * nr) * H

def gen_points(L, delta=np.pi/6, max_try=20000):
    pts = []
    for _ in range(max_try):
        if len(pts) == L: break
        x, y = np.random.uniform(0, np.pi), np.random.uniform(0, np.pi)
        if all(math.hypot(x-p[0], y-p[1]) >= delta for p in pts):
            pts.append((x, y))
    if len(pts) < L:
        raise RuntimeError(f'Cannot place {L} points with delta={delta:.3f}')
    return pts

def gen_gt(phi_l, psi_l, M=256, sigma=0.07):
    op = np.mod( np.pi*np.cos(phi_l), 2*np.pi)
    oq = np.mod(-np.pi*np.cos(psi_l), 2*np.pi)
    margin = 3 * sigma
    ax = np.linspace(-margin, 2*np.pi+margin, M, endpoint=False)
    Wp, Wq = np.meshgrid(ax, ax)
    coeff = 1 / (2*np.pi*sigma**2)
    G = sum(coeff * np.exp(-((Wp-o)**2 + (Wq-q)**2)/(2*sigma**2))
            for o, q in zip(op, oq))
    return G.astype(np.float32)

def data_generation(Training=True, condition=None, sigma=0.07, M=256):
    """Infinite generator — paper-exact distribution."""
    import random as pyrandom
    while True:
        if Training:
            L   = np.random.randint(1, 10)
            SNR = np.random.randint(-15, 25)
            P   = pyrandom.choice([16, 32])
            nt  = 16 if P == 16 else pyrandom.choice([16, 32])
        else:
            L, SNR, P, nt = condition

        Q, nr = P, nt
        F = make_F(P, nt)
        W = make_W(Q, nr)

        alpha = (np.sqrt(1/L)/np.sqrt(2)) * (np.random.randn(L) + 1j*np.random.randn(L))
        alpha = alpha[np.argsort(-np.abs(alpha))]

        pts   = gen_points(L)
        phi_l = np.array([p[0] for p in pts])
        psi_l = np.array([p[1] for p in pts])

        H = gen_channel(nr, nt, phi_l, psi_l, alpha)
        var = 10**(-SNR/10); s = np.sqrt(var/2)
        Z = s * (np.random.randn(Q, P) + 1j*np.random.randn(Q, P))
        Y = (W.view(_H).H @ H) @ F + Z

        zoom = 4 if P == 16 else 2
        import scipy.ndimage
        data = np.stack([
            scipy.ndimage.zoom(Y.real, zoom, order=0),
            scipy.ndimage.zoom(Y.imag, zoom, order=0)
        ], axis=-1).astype(np.float32)

        if Training:
            gt = gen_gt(phi_l, psi_l, M, sigma)[..., np.newaxis]
            yield data, gt
        else:
            yield data, np.stack([psi_l, phi_l]).astype(np.float32)

print('✅ Physics + data generator ready')

In [ ]:
# Cell 3 — Evaluation utilities: blob detector, angle recovery, metric
def get_detector():
    p = cv2.SimpleBlobDetector_Params()
    p.filterByColor = True; p.blobColor = 255
    p.minThreshold  = 0;    p.maxThreshold = 255
    p.filterByArea  = True; p.minArea = 1; p.maxArea = 1000
    p.filterByCircularity = p.filterByConvexity = p.filterByInertia = False
    return cv2.SimpleBlobDetector_create(p)

DETECTOR = get_detector()

def get_peaks(pred2d, L):
    """Current pipeline: OpenCV SimpleBlobDetector."""
    img = cv2.normalize(pred2d, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    kps = DETECTOR.detect(img)
    if not kps: return np.zeros((0, 2))
    coords = np.array([k.pt for k in kps])
    amps   = np.array([img[min(int(round(k.pt[1])),img.shape[0]-1),
                           min(int(round(k.pt[0])),img.shape[1]-1)] for k in kps])
    return coords[np.argsort(-amps)[:L]]

def peaks_argmax(pred2d, L):
    """Oracle: pure local-maxima finder, no blob-shape constraints at all."""
    mx  = maximum_filter(pred2d, size=5, mode='nearest')
    idx = np.argwhere(pred2d == mx)
    if len(idx) == 0: return np.zeros((0, 2))
    vals = pred2d[idx[:,0], idx[:,1]]
    top  = idx[np.argsort(-vals)[:L]]
    return np.stack([top[:,1], top[:,0]], axis=1).astype(float)  # (x=col, y=row)

def peaks2angles(peaks, sigma=0.07, M=256):
    if len(peaks) == 0: return np.array([]), np.array([])
    margin = 3 * sigma
    ext    = 2*np.pi + 2*margin
    f = -margin + (peaks.T / M) * ext
    f = np.where(f > np.pi, f - 2*np.pi, f)
    psi = np.arccos(np.clip(-f[1]/np.pi, -1, 1))
    phi = np.arccos(np.clip( f[0]/np.pi, -1, 1))
    return psi, phi

def match_and_eval(est_psi, est_phi, feat, max_deg=1.0):
    """Hungarian matching, paper-exact: source detected iff BOTH psi and phi within max_deg.
    Returns (n_detected, n_total, good_component_errors_deg, per_source_min_sinpsi)."""
    L = feat.shape[-1]
    if len(est_psi) < L: return 0, L, [], []

    est_psi = est_psi[:L]; est_phi = est_phi[:L]
    gt_pairs  = np.stack([feat[0], feat[1]], axis=1)
    est_pairs = np.stack([est_psi, est_phi], axis=1)
    dist = np.linalg.norm(gt_pairs[:,None] - est_pairs[None], axis=2)
    row_idx, col_idx = linear_sum_assignment(dist)

    n_detected = 0
    good_components = []
    miss_sinpsi = []   # sin(psi) for MISSED sources -- tests the Jacobian hypothesis
    for r, c in zip(row_idx, col_idx):
        dpsi = np.degrees(np.angle(np.exp(1j*gt_pairs[r,0])*np.exp(-1j*est_pairs[c,0])))
        dphi = np.degrees(np.angle(np.exp(1j*gt_pairs[r,1])*np.exp(-1j*est_pairs[c,1])))
        if abs(dpsi) <= max_deg and abs(dphi) <= max_deg:
            n_detected += 1
            good_components.extend([dpsi, dphi])
        else:
            miss_sinpsi.append(np.sin(gt_pairs[r,0]))

    return n_detected, L, good_components, miss_sinpsi

print('✅ Evaluation utilities ready (get_peaks, peaks_argmax, peaks2angles, match_and_eval)')

In [ ]:
# ══════════════════════════════════════════════════════════
# TEST 1 — GT-blob ceiling vs sigma (FREE, no training needed)
# ══════════════════════════════════════════════════════════
np.random.seed(0)
L = 3
N_TRIALS = 400
SIGMAS = [0.10, 0.07, 0.05, 0.03, 0.02]

print('TEST 1: GT-blob ceiling vs sigma')
print('='*60)
for sigma in SIGMAS:
    ndet, ntot, errs = 0, 0, []
    for _ in range(N_TRIALS):
        pts = gen_points(L)
        phi_l = np.array([p[0] for p in pts]); psi_l = np.array([p[1] for p in pts])
        gt = gen_gt(phi_l, psi_l, M=256, sigma=sigma)
        pk = get_peaks(gt, L)
        psi_e, phi_e = peaks2angles(pk, sigma=sigma, M=256)
        feat = np.stack([psi_l, phi_l])
        d, t, g, _ = match_and_eval(psi_e, phi_e, feat)
        ndet += d; ntot += t; errs += g
    pd_ = ndet/ntot
    rmse = np.sqrt(np.mean(np.array(errs)**2)) if errs else float('nan')
    print(f'  sigma={sigma:.2f}  →  Pd={pd_:.4f}   RMSE={rmse:.4f}°   ({ndet}/{ntot})')
print()
print('  σ কমালে Pd বাড়লে → ceiling নমনীয়, retraining দিয়ে ভাঙা যাবে')
print('  σ কমালে প্রায় না বাড়লে → ceiling structural, σ দিয়ে সমাধান নেই')

In [ ]:
# ══════════════════════════════════════════════════════════
# TEST 2 — GT-blob ceiling vs M (grid resolution)
# ══════════════════════════════════════════════════════════
np.random.seed(0)
L = 3
N_TRIALS = 400
M_VALUES = [256, 512]

print('TEST 2: GT-blob ceiling vs M (output grid size)')
print('='*60)
for M in M_VALUES:
    ndet, ntot, errs = 0, 0, []
    for _ in range(N_TRIALS):
        pts = gen_points(L)
        phi_l = np.array([p[0] for p in pts]); psi_l = np.array([p[1] for p in pts])
        gt = gen_gt(phi_l, psi_l, M=M, sigma=0.07)
        pk = get_peaks(gt, L)
        psi_e, phi_e = peaks2angles(pk, sigma=0.07, M=M)
        feat = np.stack([psi_l, phi_l])
        d, t, g, _ = match_and_eval(psi_e, phi_e, feat)
        ndet += d; ntot += t; errs += g
    pd_ = ndet/ntot
    rmse = np.sqrt(np.mean(np.array(errs)**2)) if errs else float('nan')
    print(f'  M={M}  →  Pd={pd_:.4f}   RMSE={rmse:.4f}°   ({ndet}/{ntot})')

In [ ]:
# ══════════════════════════════════════════════════════════
# TEST 3 — Missed sources: sin(psi) distribution (Jacobian hypothesis)
# ══════════════════════════════════════════════════════════
np.random.seed(0)
L = 3
N_TRIALS = 1000

all_miss_sinpsi = []
all_sinpsi = []
for _ in range(N_TRIALS):
    pts = gen_points(L)
    phi_l = np.array([p[0] for p in pts]); psi_l = np.array([p[1] for p in pts])
    gt = gen_gt(phi_l, psi_l, M=256, sigma=0.07)
    pk = get_peaks(gt, L)
    psi_e, phi_e = peaks2angles(pk)
    feat = np.stack([psi_l, phi_l])
    d, t, g, miss = match_and_eval(psi_e, phi_e, feat)
    all_miss_sinpsi += miss
    all_sinpsi += list(np.sin(psi_l))

print('TEST 3: sin(psi) — missed vs all sources')
print('='*60)
print(f'  All sources     — mean sin(ψ): {np.mean(all_sinpsi):.4f}   (median {np.median(all_sinpsi):.4f})')
if all_miss_sinpsi:
    print(f'  Missed sources  — mean sin(ψ): {np.mean(all_miss_sinpsi):.4f}   (median {np.median(all_miss_sinpsi):.4f})')
    print(f'  Missed count: {len(all_miss_sinpsi)} / {N_TRIALS*L}')
else:
    print('  কোনো miss হয়নি — sigma=0.07, M=256 তে সব ধরা পড়েছে')
print()
print('  Missed sources-এর sin(ψ) all-source গড়ের চেয়ে ছোট হলে →')
print('  Jacobian hypothesis সঠিক: psi এর extreme (0° বা 180° কাছে) সবচেয়ে বেশি miss হয়')

fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].hist(all_sinpsi, bins=30, alpha=0.6, label='All sources', density=True)
if all_miss_sinpsi:
    ax[0].hist(all_miss_sinpsi, bins=30, alpha=0.6, label='Missed sources', density=True)
ax[0].set_xlabel('sin(ψ)'); ax[0].legend(); ax[0].set_title('sin(ψ) distribution')

psi_grid = np.linspace(0.01, np.pi-0.01, 200)
jac_err_deg = np.degrees((2*np.pi/256) / (np.pi*np.sin(psi_grid)))
ax[1].plot(np.degrees(psi_grid), jac_err_deg)
ax[1].axhline(1.0, color='r', ls='--', label='1° threshold')
ax[1].set_xlabel('ψ (deg)'); ax[1].set_ylabel('angle error per pixel (deg)')
ax[1].set_title('Jacobian: pixel-quantization → angle error')
ax[1].legend(); ax[1].set_ylim(0, 5)
plt.tight_layout(); plt.show()

In [ ]:
# Cell — Load pretrained ResNet (needed for TEST 4)
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose
from tensorflow.keras.models import Model

def find_weights(filename='inf_model_007_256_resnet.h5'):
    for root in ['/kaggle/input', '/kaggle/working', '.', '/content']:
        if not os.path.isdir(root): continue
        for dp, _, files in os.walk(root):
            if filename in files: return os.path.join(dp, filename)
    return None

def build_plain_ResNet(n_blocks=64, filters=12):
    def res_conv(x, f):
        skip = x
        x = Conv2D(f,5,padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Conv2D(f,5,padding='same')(x); x = BatchNormalization()(x)
        x = Add()([x, skip]); x = Activation('relu')(x)
        return x
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(filters, 5, strides=2, padding='same')(x_in)
    for _ in range(n_blocks): x = res_conv(x, filters)
    x = Conv2DTranspose(1, 5, strides=2, padding='same')(x)
    return Model(x_in, x, name='PlainResNet-64b')

WEIGHTS_PATH = find_weights()
print(f'Weights: {WEIGHTS_PATH}')

if WEIGHTS_PATH is None:
    print('⚠️  weight file পাওয়া যায়নি — Kaggle-এ dataset attach করো')
    print('   (inf_model_007_256_resnet.h5 খুঁজছে /kaggle/input, /kaggle/working, ., /content)')
else:
    model = build_plain_ResNet(n_blocks=64, filters=12)
    model.load_weights(WEIGHTS_PATH)
    print(f'✅ Loaded — {model.count_params():,} params')

In [ ]:
# ══════════════════════════════════════════════════════════
# TEST 4 — Diffusion GO/NO-GO: blob vs sharpened vs argmax
# ══════════════════════════════════════════════════════════
# hypothesis: MSE -> posterior mean -> blurry output at low SNR -> blob detector fails.
# If argmax/sharpened >> blob  => extraction/sharpness is the bottleneck (diffusion may help)
# If all three ~equal          => peaks are in the WRONG place, not blurry (diffusion won't help)

assert WEIGHTS_PATH is not None, 'আগে pretrained weight load করো (আগের cell)'

N_TRIALS = 300
L = 3
SNRS = [-10, -5, 0, 10, 25]

print('TEST 4: Diffusion go/no-go — same ResNet output, 3 extraction methods')
print('='*78)
print(f'{"SNR":>5} | {"blob (current)":>15} | {"sharpened p^4":>15} | {"argmax (oracle)":>16}')
print('-'*78)

table = {}
for snr in SNRS:
    gen = data_generation(Training=False, condition=(L, snr, 16, 16))
    acc = {'blob':[0,0], 'sharp':[0,0], 'argmax':[0,0]}

    for _ in range(N_TRIALS):
        data, feat = next(gen)
        pred = model(tf.expand_dims(data, 0), training=False)[0,:,:,0].numpy()
        pn = pred - pred.min()
        pn = pn / (pn.max() + 1e-9)   # normalize to [0,1]

        for key, p_in, finder in [
            ('blob',   pn,     lambda p: get_peaks(p, L)),
            ('sharp',  pn**4,  lambda p: get_peaks(p, L)),
            ('argmax', pn,     lambda p: peaks_argmax(p, L)),
        ]:
            psi_e, phi_e = peaks2angles(finder(p_in))
            d, t, g, _ = match_and_eval(psi_e, phi_e, feat)
            acc[key][0] += d; acc[key][1] += t

    row = {k: acc[k][0]/acc[k][1] for k in acc}
    table[snr] = row
    print(f'{snr:>5} | {row["blob"]:>15.4f} | {row["sharp"]:>15.4f} | {row["argmax"]:>16.4f}')

print('='*78)
print()
print('সিদ্ধান্তের নিয়ম:')
print('  argmax বা sharpened, blob-এর চেয়ে +0.05 Pd বেশি (বিশেষত low SNR-এ)')
print('     → তীক্ষ্ণতা/extraction-ই বাধা → sharpening (σ↓/focal loss) আগে, তারপর দরকার হলে diffusion')
print('  তিনটে ±0.02-এর মধ্যে')
print('     → peak ভুল জায়গায় (noise), ঝাপসা নয় → diffusion বাদ, set prediction-এ যাও')

In [ ]:
# Cell — Summary plot: TEST 4 across SNR
snrs = sorted(table.keys())
fig, ax = plt.subplots(figsize=(8,5))
for key, lbl, marker in [('blob','Blob detector (current)','o-'),
                          ('sharp','Sharpened p^4','s--'),
                          ('argmax','Argmax (oracle)','^:')]:
    ax.plot(snrs, [table[s][key] for s in snrs], marker, label=lbl, lw=2, ms=7)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Pd')
ax.set_title('Diffusion Go/No-Go — extraction method comparison (same ResNet)')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig('/kaggle/working/diffusion_gonogo.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: diffusion_gonogo.png')